# 01 — Exploratory Data Analysis & Data-Quality Audit
**Capstone:** Explainable and Efficient Machine Learning for Short-Term Solar Power Forecasting

**Purpose of this notebook:** *audit* the raw data before any modelling. It does not clean or
transform anything on disk; it documents what is wrong or odd so that the preprocessing step can
handle each issue deliberately (and so I can justify every choice in the viva).

**Data:** Kaggle "Solar Power Generation Data" — 2 plants in India, ~34 days, 15-minute intervals.
Each plant has a *Generation* file (one row per inverter per timestamp) and a *Weather Sensor* file
(one row per timestamp, from a single plant-level sensor).

**Structure:** 1 Load & basic checks → 2 Timestamp parsing → 3 Gaps & alignment → 4 DC/AC ratio →
5 Physical sanity checks → 6 Plots → 7 Summary.

## 0. Setup
*Decision:* paths are relative to the `notebooks/` folder and can be overridden with environment
variables, so the notebook runs unchanged on any machine. All figures go to `results/figures/`.
A small `note()` helper records each key finding as it is computed, so the final summary table is
generated from the data rather than typed by hand.

In [ ]:
import os, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")

RAW_DIR = Path(os.environ.get("SOLAR_RAW_DIR", "../data/raw"))
FIG_DIR = Path(os.environ.get("SOLAR_FIG_DIR", "../results/figures"))
TAB_DIR = Path(os.environ.get("SOLAR_TAB_DIR", "../results/tables"))
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

PLANTS = (1, 2)
FREQ = "15min"
PLANT_COLOR = {1: "#1f77b4", 2: "#d95f02"}

def savefig(fig, name):
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved -> {FIG_DIR / name}")

findings = []          # (section, message) collected through the notebook
def note(section, msg):
    findings.append((section, msg))
    print("•", msg)

## 1. Load the four CSVs and run basic checks
*Decision:* `DATE_TIME` is read as a plain **string** on purpose. If pandas guessed the format it
could silently swap day and month (`05-06-2020` is 5 June or 6 May?). I parse it explicitly in §2.
Checks per file: shape, dtypes, missing values, and duplicates (both fully identical rows and
duplicated *keys*, i.e. the same timestamp + source appearing twice with possibly different values).

In [ ]:
def find_csv(plant, kind):
    # kind is 'generation' or 'weather'; matches e.g. Plant_1_Generation_Data.csv
    pat = re.compile(rf"plant[_ ]?{plant}.*{kind}", re.I)
    hits = [p for p in RAW_DIR.glob("*.csv") if pat.search(p.name)]
    if len(hits) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{kind}' CSV for plant {plant} in {RAW_DIR.resolve()}, found {hits}")
    return hits[0]

raw = {}
for p in PLANTS:
    raw[(p, "gen")] = pd.read_csv(find_csv(p, "generation"), dtype={"DATE_TIME": "string"})
    raw[(p, "wx")]  = pd.read_csv(find_csv(p, "weather"),    dtype={"DATE_TIME": "string"})

rows = []
for (p, k), df in raw.items():
    rows.append({
        "plant": p, "file": "generation" if k == "gen" else "weather",
        "rows": df.shape[0], "cols": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_keys(DATE_TIME+SOURCE_KEY)": int(df.duplicated(["DATE_TIME", "SOURCE_KEY"]).sum()),
        "n_sources": df["SOURCE_KEY"].nunique(),
    })
overview = pd.DataFrame(rows)
display(overview)
overview.to_csv(TAB_DIR / "eda_overview.csv", index=False)

for (p, k), df in raw.items():
    print(f"\n--- Plant {p} {k}: dtypes / missing per column ---")
    display(pd.DataFrame({"dtype": df.dtypes.astype(str), "n_missing": df.isna().sum()}).T)

In [ ]:
for (p, k), df in raw.items():
    name = "generation" if k == "gen" else "weather"
    r = overview[(overview.plant == p) & (overview.file == name)].iloc[0]
    if r.missing_cells:
        note("1 load", f"Plant {p} {name}: {r.missing_cells} missing cells")
    if r.duplicate_rows or r["duplicate_keys(DATE_TIME+SOURCE_KEY)"]:
        note("1 load", f"Plant {p} {name}: {r.duplicate_rows} duplicate rows, "
                       f"{r['duplicate_keys(DATE_TIME+SOURCE_KEY)']} duplicate keys")
print("Plants differ in inverter count?", {p: raw[(p, 'gen')]['SOURCE_KEY'].nunique() for p in PLANTS})
print("\nGeneration describe (numeric):")
for p in PLANTS:
    print(f"Plant {p}"); display(raw[(p, 'gen')].drop(columns=["PLANT_ID"]).describe().T.round(2))

## 2. Parse `DATE_TIME` carefully
The two plants are known to be exported with different conventions, but I **verify rather than assume**:

1. Inspect sample strings per file.
2. Detect the format with strict regexes. If a file matches *neither* pattern (or a mix), the code
   raises an error instead of guessing.
3. Parse with an explicit `format=`.
4. Validate: all four files must span roughly the same ~34 days. As a demonstration of the risk, I
   also show how many values would fail (`NaT`) if a day-first file were mis-read as month-first.

In [ ]:
FORMATS = {
    r"^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$":          "%d-%m-%Y %H:%M",     # e.g. 15-05-2020 00:15
    r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$":    "%Y-%m-%d %H:%M:%S",  # e.g. 2020-05-15 00:15:00
}

def detect_format(s):
    for pat, fmt in FORMATS.items():
        if s.str.match(pat).fillna(False).all():
            return fmt
    raise ValueError(f"Unrecognised/mixed DATE_TIME format. Samples: {s.sample(5, random_state=0).tolist()}")

data, rows = {}, []
for (p, k), df in raw.items():
    fmt = detect_format(df["DATE_TIME"])
    ts = pd.to_datetime(df["DATE_TIME"], format=fmt)
    # what a wrong (month-first) reading of a day-first file would do:
    wrong_nat = (pd.to_datetime(df["DATE_TIME"], format="%m-%d-%Y %H:%M", errors="coerce").isna().mean()
                 if fmt.startswith("%d") else np.nan)
    data[(p, k)] = df.assign(ts=ts)
    rows.append({"plant": p, "file": "generation" if k == "gen" else "weather",
                 "sample": df["DATE_TIME"].iloc[0], "format": fmt,
                 "start": ts.min(), "end": ts.max(), "span_days": round((ts.max() - ts.min()).total_seconds() / 86400, 2),
                 "NaT_if_misread_month_first": wrong_nat})
fmt_table = pd.DataFrame(rows)
display(fmt_table)
assert fmt_table["span_days"].between(30, 40).all(), "Unexpected date span - check parsing!"
assert all(data[k]["ts"].notna().all() for k in data)

In [ ]:
if (fmt_table["format"].nunique() > 1):
    note("2 parse", "DATE_TIME formats DIFFER across files: " +
         "; ".join(f"P{r.plant} {r.file}={r.format}" for r in fmt_table.itertuples()))
note("2 parse", "All files parsed with explicit formats; span "
     f"{fmt_table.start.min():%Y-%m-%d} -> {fmt_table.end.max():%Y-%m-%d}")
# are all timestamps exactly on the 15-minute grid?
for (p, k), df in data.items():
    off = ((df.ts.dt.minute % 15 != 0) | (df.ts.dt.second != 0)).sum()
    if off:
        note("2 parse", f"Plant {p} {k}: {off} timestamps not on the 15-min grid")

## 3. Timestamp gaps and generation ↔ weather alignment
*Why it matters:* forecasting models use lagged values (e.g. "power 15/30/60 min ago"). If rows are
missing, a `shift(1)` silently becomes "the previous *available* row", which may be hours earlier.
Features must therefore be built on a **complete 15-min grid**, and the generation and weather
tables must be joinable on timestamp.

*Method:* build the full expected 15-min grid from min to max timestamp, and compare. Generation is
checked at plant level (any inverter reporting) and per inverter (each inverter's own completeness).

In [ ]:
rows = []
for (p, k), df in data.items():
    u = pd.DatetimeIndex(np.sort(df.ts.unique()))
    full = pd.date_range(u.min(), u.max(), freq=FREQ)
    rows.append({"plant": p, "file": "generation" if k == "gen" else "weather",
                 "expected_15min": len(full), "observed_unique": len(u),
                 "missing": len(full.difference(u)),
                 "missing_pct": round(100 * len(full.difference(u)) / len(full), 2),
                 "longest_gap": pd.Series(u).diff().max()})
gaps = pd.DataFrame(rows)
display(gaps)
gaps.to_csv(TAB_DIR / "eda_timestamp_gaps.csv", index=False)
for r in gaps.itertuples():
    if r.missing:
        note("3 gaps", f"Plant {r.plant} {r.file}: {r.missing} of {r.expected_15min} 15-min slots missing "
                       f"({r.missing_pct}%), longest gap {r.longest_gap}")

In [ ]:
# Per-inverter completeness and number of inverters reporting per timestamp
for p in PLANTS:
    g = data[(p, "gen")]
    full_n = len(pd.date_range(g.ts.min(), g.ts.max(), freq=FREQ))
    per_inv = g.groupby("SOURCE_KEY")["ts"].nunique().rename("n_timestamps").to_frame()
    per_inv["completeness_%"] = (100 * per_inv.n_timestamps / full_n).round(1)
    n_per_ts = g.groupby("ts")["SOURCE_KEY"].nunique()
    print(f"Plant {p}: inverter completeness min/median/max = "
          f"{per_inv['completeness_%'].min()} / {per_inv['completeness_%'].median()} / {per_inv['completeness_%'].max()} %")
    print(f"          inverters reporting per timestamp (value counts, top):")
    display(n_per_ts.value_counts().sort_index(ascending=False).head(6).to_frame("n_timestamps").T)
    if per_inv["completeness_%"].min() < 99:
        note("3 gaps", f"Plant {p}: some inverters are incomplete (min completeness "
                       f"{per_inv['completeness_%'].min()}%) -> aggregate with mean, not sum, or impute")

In [ ]:
# Do generation and weather share the same timestamps?
rows = []
for p in PLANTS:
    g, w = set(data[(p, "gen")].ts), set(data[(p, "wx")].ts)
    rows.append({"plant": p, "gen_ts": len(g), "wx_ts": len(w), "in_both": len(g & w),
                 "gen_only": len(g - w), "wx_only": len(w - g),
                 "gen_rows_with_weather_%": round(100 * data[(p, "gen")].ts.isin(w).mean(), 2)})
    if g - w or w - g:
        note("3 align", f"Plant {p}: gen/weather timestamps do not fully align "
                        f"(gen-only={len(g - w)}, weather-only={len(w - g)}); use an inner join on timestamp")
align = pd.DataFrame(rows); display(align)
align.to_csv(TAB_DIR / "eda_alignment.csv", index=False)

## 4. DC vs AC power ratio (unit / scale check)
*Physics:* an inverter converts DC (from the panels) to AC (to the grid) with ~95–98 % efficiency, so
**DC/AC should be ≈ 1.02–1.05** and must be ≥ 1. A ratio far from that, or very different between two
plants of the same kind, signals a **unit/scale error** (e.g. DC logged in different units, or a
×10 factor) rather than real behaviour.

*Method:* ratio only where both DC and AC are > 0 (a ratio at night is 0/0). Flag a plant if its
median ratio falls outside [1.0, 1.15], or if the two plants' medians differ by more than 1.5×.

In [ ]:
ratios = {}
for p in PLANTS:
    g = data[(p, "gen")]
    m = (g.DC_POWER > 0) & (g.AC_POWER > 0)
    ratios[p] = (g.loc[m, "DC_POWER"] / g.loc[m, "AC_POWER"])

summary = pd.DataFrame({p: r.describe(percentiles=[.05, .5, .95]) for p, r in ratios.items()}).T.round(3)
display(summary)
summary.to_csv(TAB_DIR / "eda_dc_ac_ratio.csv")

meds = {p: ratios[p].median() for p in PLANTS}
for p in PLANTS:
    if not 1.0 <= meds[p] <= 1.15:
        note("4 ratio", f"Plant {p}: median DC/AC = {meds[p]:.2f} (expected ~1.02-1.05) -> LIKELY UNIT/SCALE ISSUE in DC_POWER or AC_POWER")
    else:
        print(f"Plant {p}: median DC/AC = {meds[p]:.3f} -> plausible")
if max(meds.values()) / min(meds.values()) > 1.5:
    note("4 ratio", f"DC/AC medians differ between plants ({meds[1]:.2f} vs {meds[2]:.2f}) -> do not pool DC_POWER across plants without rescaling")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, p in zip(axes, PLANTS):
    sns.histplot(ratios[p].clip(upper=ratios[p].quantile(.999)), bins=80, ax=ax, color=PLANT_COLOR[p])
    ax.axvline(1.0, color="k", ls="--", lw=1); ax.set_title(f"Plant {p}: DC/AC (median {meds[p]:.2f})")
    ax.set_xlabel("DC_POWER / AC_POWER")
fig.suptitle("DC/AC power ratio per plant (rows with DC>0 and AC>0)", y=1.02)
savefig(fig, "00_dc_ac_ratio.png")

## 5. Physical sanity checks
Cheap checks that catch sensor/logging faults:
- **Negative** power or irradiation (should not occur).
- **Night-time power**: `AC_POWER > 0` when irradiation is 0 (should be ~0).
- **Daytime zeros**: irradiation clearly > 0 but plant output = 0 (inverter outage / curtailment —
  these look like "huge forecast errors" to a model but are not weather-driven).
- **`TOTAL_YIELD` must never decrease** per inverter (it is a cumulative counter). It is
  cumulative, so it is a poor/leaky forecasting feature — I will not use it as a predictor.

Plant-level frame used from here on: *mean* AC/DC across inverters per timestamp (robust to a missing inverter,
unlike a sum), inner-joined to weather on timestamp.

In [ ]:
WX = ["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]

def plant_frame(p):
    g, w = data[(p, "gen")], data[(p, "wx")]
    a = g.groupby("ts").agg(AC_mean=("AC_POWER", "mean"), DC_mean=("DC_POWER", "mean"),
                            n_inv=("SOURCE_KEY", "nunique"))
    return a.join(w.groupby("ts")[WX].mean(), how="inner")

pf = {p: plant_frame(p) for p in PLANTS}

rows = []
for p in PLANTS:
    g, f = data[(p, "gen")], pf[p]
    tot = g.sort_values("ts").groupby("SOURCE_KEY")["TOTAL_YIELD"].apply(lambda s: int((s.diff() < 0).sum()))
    rows.append({"plant": p,
                 "neg_AC": int((g.AC_POWER < 0).sum()), "neg_DC": int((g.DC_POWER < 0).sum()),
                 "neg_irradiation": int((data[(p, 'wx')].IRRADIATION < 0).sum()),
                 "night_ts_AC_pos": int(((f.IRRADIATION == 0) & (f.AC_mean > 0)).sum()),
                 "day_ts_AC_zero": int(((f.IRRADIATION > 0.1) & (f.AC_mean == 0)).sum()),
                 "TOTAL_YIELD_decreases": int(tot.sum())})
sanity = pd.DataFrame(rows); display(sanity)
sanity.to_csv(TAB_DIR / "eda_sanity_checks.csv", index=False)
for r in sanity.itertuples():
    if r.neg_AC or r.neg_DC or r.neg_irradiation:
        note("5 sanity", f"Plant {r.plant}: negative power/irradiation values present")
    if r.night_ts_AC_pos > 0:
        note("5 sanity", f"Plant {r.plant}: {r.night_ts_AC_pos} night timestamps (irradiation=0) with AC>0")
    if r.day_ts_AC_zero > 0:
        note("5 sanity", f"Plant {r.plant}: {r.day_ts_AC_zero} daytime timestamps (irradiation>0.1) with plant AC=0 "
                         f"-> outages/curtailment, consider flagging not forecasting")
    if r.TOTAL_YIELD_decreases:
        note("5 sanity", f"Plant {r.plant}: TOTAL_YIELD decreases {r.TOTAL_YIELD_decreases} times (counter reset?)")

## 6. Visual exploration
### 6.1 Daily generation curves
Plant-level mean AC power per inverter vs time of day; one thin line per day and a bold median.
*What I look for:* the bell shape, consistent sunrise/sunset times, cloudy days (dips), and days that are
truncated by missing data. The lower panel shows total daily energy per inverter
(kW × 0.25 h per 15-min step) to spot unusually low days.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), gridspec_kw={"height_ratios": [2, 1]})
cov_rows = []
for j, p in enumerate(PLANTS):
    f = pf[p].copy()
    f["date"], f["tod"] = f.index.normalize(), f.index.hour + f.index.minute / 60
    ax = axes[0, j]
    for d, grp in f.groupby("date"):
        ax.plot(grp.tod, grp.AC_mean, color="grey", alpha=.35, lw=.8)
    med = f.groupby("tod").AC_mean.median()
    ax.plot(med.index, med.values, color=PLANT_COLOR[p], lw=2.5, label="median over days")
    ax.set(title=f"Plant {p}: daily AC power curves", xlabel="hour of day", ylabel="mean AC_POWER per inverter (kW)")
    ax.legend()
    daily = f.groupby("date").AC_mean.sum() * 0.25         # kWh per inverter
    axes[1, j].bar(daily.index, daily.values, color=PLANT_COLOR[p], width=.8)
    axes[1, j].set(title=f"Plant {p}: daily energy per inverter", ylabel="kWh"); axes[1, j].tick_params(axis="x", rotation=45)
    cov = f.groupby("date").size().rename("n_intervals"); cov_rows.append(cov.to_frame().assign(plant=p))
fig.tight_layout()
savefig(fig, "01_daily_generation_curves.png")

cov = pd.concat(cov_rows)
short = cov[cov.n_intervals < 96 * 0.95]
print("Days with <95% of the 96 daily intervals present (after joining to weather):")
display(short if len(short) else "none")
if len(short):
    note("6 plots", f"{len(short)} plant-days have <95% of 96 intervals (partial days: first/last day or outages)")

### 6.2 Irradiation vs AC power
This is the core physical relationship the model must learn: output rises ~linearly with irradiation
(then saturates/derates when hot). Colour = hour of day, to reveal morning/afternoon asymmetry
(e.g. soiling, shading, or sensor lag). A tight band means irradiation is a very strong predictor;
scatter around it is what temperature and time features must explain.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
for ax, p in zip(axes, PLANTS):
    f = pf[p]
    sc = ax.scatter(f.IRRADIATION, f.AC_mean, c=f.index.hour + f.index.minute / 60, s=6, alpha=.5, cmap="viridis")
    ax.set(title=f"Plant {p}", xlabel="IRRADIATION (kW/m²)", ylabel="mean AC_POWER per inverter (kW)")
    r = f[["IRRADIATION", "AC_mean"]].corr().iloc[0, 1]
    ax.text(.03, .93, f"Pearson r = {r:.3f}", transform=ax.transAxes)
fig.colorbar(sc, ax=axes, label="hour of day")
savefig(fig, "02_irradiation_vs_ac.png")

### 6.3 Module vs ambient temperature
Modules run hotter than the air (heated by irradiation). Points above the dashed 1:1 line are expected
in daylight; module ≈ ambient at night. High temperature *reduces* panel efficiency, which is why
`MODULE_TEMPERATURE` is a useful (but correlated-with-irradiation) feature.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, p in zip(axes, PLANTS):
    f = pf[p]
    sc = ax.scatter(f.AMBIENT_TEMPERATURE, f.MODULE_TEMPERATURE, c=f.IRRADIATION, s=6, alpha=.6, cmap="plasma")
    lim = [min(f.AMBIENT_TEMPERATURE.min(), f.MODULE_TEMPERATURE.min()), max(f.AMBIENT_TEMPERATURE.max(), f.MODULE_TEMPERATURE.max())]
    ax.plot(lim, lim, "k--", lw=1, label="module = ambient")
    ax.set(title=f"Plant {p}", xlabel="AMBIENT_TEMPERATURE (°C)", ylabel="MODULE_TEMPERATURE (°C)"); ax.legend(loc="lower right")
    delta = (f.MODULE_TEMPERATURE - f.AMBIENT_TEMPERATURE)[f.IRRADIATION > .5]
    print(f"Plant {p}: median (module - ambient) when irradiation>0.5 = {delta.median():.1f} °C; "
          f"module<ambient in daylight for {(delta < 0).mean():.1%} of such rows")
fig.colorbar(sc, ax=axes, label="IRRADIATION")
savefig(fig, "03_module_vs_ambient_temp.png")

### 6.4 Correlation heatmap
Pearson correlation of the plant-level variables (all hours, i.e. what a model would see, so night-time
zeros are included and inflate correlations with irradiation). `TOTAL_YIELD`/`DAILY_YIELD` are excluded:
they are cumulative counters, not instantaneous drivers. Note the expected **high collinearity**
(irradiation ↔ module temperature ↔ power) — relevant later for SHAP interpretation and for linear
models.

In [ ]:
cols = ["AC_mean", "DC_mean", "IRRADIATION", "MODULE_TEMPERATURE", "AMBIENT_TEMPERATURE"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, p in zip(axes, PLANTS):
    sns.heatmap(pf[p][cols].corr(), annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="coolwarm", ax=ax, cbar=p == 2)
    ax.set_title(f"Plant {p}")
fig.tight_layout()
savefig(fig, "04_correlation_heatmap.png")

### 6.5 Per-inverter output — spotting underperforming inverters
*Method:* restrict to **daylight** timestamps (plant irradiation > 0.1) so that night zeros do not
dilute differences; take each inverter's mean AC power and divide by the **median across inverters**
of the same plant. A ratio of 1.0 means "typical"; **< 0.90** is flagged as a candidate
underperformer (the 10 % threshold is a pragmatic first cut, to be refined with the residual-based
flagging later). Using the mean per row (not a sum) avoids penalising an inverter just for missing rows.
The second figure shows *when* the deficit happens (a persistent fault vs. a few bad days).
Inverter IDs are shortened to `P<plant>-I<nn>`; the mapping is saved in `results/tables/inverter_summary.csv`.

In [ ]:
inv_tables, inv_day = [], {}
for p in PLANTS:
    g = data[(p, "gen")]
    day_ts = pf[p].index[pf[p].IRRADIATION > 0.1]
    gd = g[g.ts.isin(day_ts)].copy()
    keys = sorted(g.SOURCE_KEY.unique())
    ids = {k: f"P{p}-I{i + 1:02d}" for i, k in enumerate(keys)}
    gd["inv"] = gd.SOURCE_KEY.map(ids)
    m = gd.groupby("inv").AC_POWER.agg(mean_daylight_AC="mean", n_rows="size")
    m["rel_to_median"] = m.mean_daylight_AC / m.mean_daylight_AC.median()
    m["flagged"] = m.rel_to_median < 0.90
    m = m.reset_index().assign(plant=p, SOURCE_KEY=m.index.map({v: k for k, v in ids.items()}))
    inv_tables.append(m)
    gd["date"] = gd.ts.dt.normalize()
    d = gd.groupby(["inv", "date"]).AC_POWER.mean().unstack("date")
    inv_day[p] = d / d.median(axis=0)            # relative to that day's median inverter
inv = pd.concat(inv_tables)[["plant", "inv", "SOURCE_KEY", "mean_daylight_AC", "rel_to_median", "flagged", "n_rows"]]
inv.to_csv(TAB_DIR / "inverter_summary.csv", index=False)
display(inv[inv.flagged].round(3) if inv.flagged.any() else "No inverter below 90% of the plant median")
for p in PLANTS:
    fl = inv[(inv.plant == p) & inv.flagged]
    spread = inv[inv.plant == p].rel_to_median
    note("6 inverters", f"Plant {p}: {len(fl)} of {len(spread)} inverters <90% of median daylight output "
                        f"({', '.join(fl.inv) or 'none'}); range {spread.min():.2f}-{spread.max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, p in zip(axes, PLANTS):
    d = inv[inv.plant == p].sort_values("rel_to_median")
    ax.bar(d.inv, d.rel_to_median, color=np.where(d.flagged, "#c0392b", PLANT_COLOR[p]))
    ax.axhline(1, color="k", lw=1); ax.axhline(.9, color="#c0392b", ls="--", lw=1, label="0.90 threshold")
    ax.set(title=f"Plant {p}: mean daylight AC power relative to median inverter", ylabel="relative output"); ax.tick_params(axis="x", rotation=90); ax.legend()
fig.tight_layout()
savefig(fig, "05_inverter_relative_output.png")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, p in zip(axes, PLANTS):
    d = inv_day[p].copy(); d.columns = [c.strftime("%m-%d") for c in d.columns]
    sns.heatmap(d, cmap="RdBu", center=1, vmin=.5, vmax=1.5, ax=ax, cbar_kws={"label": "output vs day-median inverter"})
    ax.set_title(f"Plant {p}: inverter × day"); ax.set_xlabel("date (MM-DD)")
fig.tight_layout()
savefig(fig, "06_inverter_day_heatmap.png")

## 7. Findings recorded by this notebook (auto-generated)
Every `note()` call above is collected here and saved to `results/tables/eda_findings.csv`.

In [ ]:
find_df = pd.DataFrame(findings, columns=["section", "finding"])
find_df.to_csv(TAB_DIR / "eda_findings.csv", index=False)
with pd.option_context("display.max_colwidth", 200):
    display(find_df)
print("Figures written:"); [print(" ", p.name) for p in sorted(FIG_DIR.glob("*.png"))]

## 8. Summary: key findings & preprocessing to-do list
> The numbers behind each item are in the auto-generated table above and in `results/tables/`.
> Items are the data-quality issues this dataset is known to raise; **tick each one only after
> confirming it against the output above.**

**Structure**
- Two plants, each with a *Generation* file (many inverters × timestamps) and a *Weather* file (one sensor × timestamps).
  Weather and generation are at plant level for irradiation/temperature, so **weather values are identical for all inverters** of a plant.

**Issues to handle in preprocessing** (and why)
1. **Date formats (§2):** parse each file with its own *explicit* format; never use `dayfirst` guessing. Re-check the span after parsing.
2. **Missing 15-min slots (§3):** reindex to a complete 15-min grid before creating lag/rolling features; decide per gap length:
   interpolate short gaps, drop windows spanning long gaps. Never use `shift()` on a gappy index.
3. **Generation ↔ weather alignment (§3):** join on timestamp with an inner join; report rows lost.
4. **DC/AC scale (§4):** if a plant's DC/AC ratio is far from ~1.03 (or the plants disagree), `DC_POWER` has a scale/unit problem.
   **Decision:** use **`AC_POWER` as the forecasting target** (power delivered to the grid), and exclude `DC_POWER` as a predictor
   (it is near-collinear with AC and would be unavailable/leaky at forecast time anyway).
5. **Target aggregation:** forecast **plant-level** (mean or sum over inverters) for the main models; keep inverter-level data for the
   underperformance/residual flagging stage. Use the *mean per inverter* if inverter counts per timestamp vary.
6. **Night and zero-output periods (§5):** night values are trivially ~0; consider training/evaluating on daylight only
   (or report both) so metrics are not inflated by easy zeros. Daytime zeros with high irradiation are outages — mark them, do not treat as weather signal.
7. **Cumulative counters:** `DAILY_YIELD` / `TOTAL_YIELD` are not usable predictors (cumulative, target-leaking) — drop.
8. **Underperforming inverters (§6.5):** candidates flagged by the 0.90 rule are *inputs to the later residual-based flagging*, not labels;
   keep them in the data but record their IDs. Decide whether to exclude them when training the plant-level model.
9. **Collinearity (§6.4):** irradiation, module temperature and power are strongly correlated → expect shared credit in SHAP;
   linear-model coefficients will be unstable. Consider reporting SHAP with and without module temperature.
10. **Small data (~34 days):** use a **chronological** train/validation/test split (no random shuffling, which leaks neighbouring
    15-min intervals) and keep whole days together; a full-year seasonality cannot be learned, so state this limitation in the report.